# GSE173958: Lineage-Validated scTour-MIC Inference



## Dataset

This tutorial follows the scTour-style tutorial layout: dataset, model training, inference, visualization, and robustness.

`GSE173958` is the strongest validation dataset for scMIC because it contains paired single-cell transcriptomes and CRISPR lineage tracing in metastatic PDAC. Primary tumor cells can be mapped to liver, lung, macrometastasis, and CTC targets. Aggressive clone labels provide the closest available gold standard for MIC prediction.

This notebook summarizes the real M1 run included in the repository. The full computation is handled by `scripts/run_gse173958_sctour_validation.py`, which trains scTour, performs organ-specific UOT, parses macsGESTALT lineage labels, validates MIC scores, and ranks genes with SHAP.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(repo_root))

import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import Image, display



## Data Download

Inspect the GEO supplementary file list first. The full dataset is large, so the command below is shown as a controlled download entry point. Run without `--dry-run` when you are ready to download the real matrices.


In [ ]:
# From the repository root:
# !python scripts/download_geo.py --dry-run --from-filelist --gse GSE173958
# !python scripts/download_geo.py --from-filelist --gse GSE173958

raw_dir = repo_root / 'data' / 'raw' / 'GSE173958'
raw_dir


## Data Loading

The example below uses mouse M1 primary tumor as the source and liver/lung/macrometastasis samples as metastatic targets. Downsampling keeps the tutorial fast; remove it for full analysis.


In [ ]:
# Uncomment after downloading GSE173958 files.
# pt_triplet = find_10x_triplet(raw_dir, 'M1-PT')
# liver_triplet = find_10x_triplet(raw_dir, 'M1-Liver')
# lung_triplet = find_10x_triplet(raw_dir, 'M1-Lung')
# met_triplet = find_10x_triplet(raw_dir, 'M1-Met')
#
# primary = downsample_cells(read_10x_mtx(*pt_triplet), n_cells=2500)
# metastases = {
#     'liver': downsample_cells(read_10x_mtx(*liver_triplet), n_cells=1500),
#     'lung': downsample_cells(read_10x_mtx(*lung_triplet), n_cells=1500),
#     'macromet': downsample_cells(read_10x_mtx(*met_triplet), n_cells=1500),
# }
# primary.shape, {k: v.shape for k, v in metastases.items()}


## Model Training

scMIC learns a scTour latent space and pseudotime-like metastatic-state axis, then performs organ-specific unbalanced optimal transport in that latent space. The current implementation uses `sctour_MIC_score` as the primary MIC score and OT-derived scores as organotropic mapping signals.


In [ ]:
# Full server run used for the repository results:
# !python scripts/run_gse173958_sctour_validation.py \
#     --raw-dir data/raw/GSE173958 \
#     --max-cells-per-sample 4000 \
#     --epochs 30


## MIC Inference

The key output is a primary-cell score table with scTour-MIC scores, organ-specific OT scores, lineage labels, EMT validation scores, and SHAP-ready high-MIC labels.


In [ ]:
scores = pd.read_csv(repo_root / 'data/processed/GSE173958_m1_sctour_ot_mic_scores.csv', index_col=0)
scores[['sctour_MIC_score', 'transport_pan_score', 'met', 'liver', 'lung', 'predicted_site', 'aggressive_clone']].head()


## Visualization

Use rank plots and organ-score distributions to inspect whether top primary cells have a focused or broad metastatic destination profile.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
scores['sctour_MIC_score'].sort_values(ascending=False).reset_index(drop=True).plot(ax=ax)
ax.set_xlabel('Primary cells ranked by scTour-MIC score')
ax.set_ylabel('scTour-MIC score')
ax.set_title('GSE173958 primary-cell MIC ranking')
plt.show()

display(Image(filename=repo_root / 'figures/GSE173958_sample_pseudotime_distribution.png'))
display(Image(filename=repo_root / 'figures/GSE173958_pseudotime_lineage_trend.png'))
display(Image(filename=repo_root / 'figures/GSE173958_sctour_backbone_by_sample.png'))
display(Image(filename=repo_root / 'figures/GSE173958_lineage_enrichment.png'))


## Lineage Validation

The workflow parses macsGESTALT `*.stats.txt.gz` files and maps read-level lineage edits back to 10x cell barcodes. For M1, the dominant metastatic lineage group is used as the aggressive-clone gold standard.


In [ ]:
metrics = pd.read_csv(repo_root / 'data/processed/GSE173958_lineage_validation_metrics.csv')
metrics


## SHAP Gene Ranking

After scTour-MIC inference, train an interpretable model to distinguish high-MIC and low-MIC primary cells. SHAP values prioritize candidate genes; combine them with lineage enrichment for the final ranked list.


In [ ]:
gene_ranking = pd.read_csv(repo_root / 'data/processed/GSE173958_shap_gene_ranking.csv')
gene_ranking.head(30)

display(Image(filename=repo_root / 'figures/GSE173958_shap_top_genes.png'))


## Robustness

Repeat inference across random seeds, scTour latent dimensions/epochs, `epsilon`, `rho`, and `top_k`. Stable MICs should recur in the top-ranked primary cells and remain enriched in aggressive clones.


In [ ]:
# Example robustness command:
# for eps in 0.04 0.08 0.12; do
#   python scripts/run_gse173958_sctour_validation.py --epsilon $eps --outdir data/processed/eps_$eps
# done
